In [3]:
using Revise
using Statistics

using Pkg; Pkg.activate(".")

using PythonPlot
using JLD2
using CSV
using DataFrames
using NCDatasets
using Printf

include("./read_lidar.jl")
using .read_lidar
using .read_lidar.stare
using .read_vecnav: read_vecnav_dict
import .chunks
read_stare_time  = Main.chunks.read_stare_time
read_stare_chunk = Main.chunks.read_stare_chunk
fit_offset = Main.chunks.fit_offset

include("./timing_lidar.jl")
using .timing_lidar

  Activating project at `~/Projects/lidar/ASTRAL2024`


In [40]:
# load lidar timing and indexing data

# ic         all lidar chunk indices
# dtime_st   lidar chunk start times
# icvn       chunk indices with Vn data

# load:

FileInds = load("file_beam_inds.jld2")
bigind_file_ends   = FileInds["bigind_file_end"] # 991, one per file
bigind_file_starts = FileInds["bigind_file_start"]

LidarDt = load("lidar_dt.jld2")
iens               = LidarDt[ "ien"            ] # 5938 chunk start indices
ists               = LidarDt[ "ist"            ] #            end
# LidarDt["dtime"] # 3174132, one per second
dtime_st = LidarDt["dtime"][ists]
dtime_en = LidarDt["dtime"][iens]
ic = eachindex(iens)

Vn = read_vecnav_dict()
icvn = findfirst( dtime_st .>= Vn[:vndt][1] ):findlast( dtime_en .<= Vn[:vndt][end] )

# sync offset and quality data
synclog = CSV.read("data/vn_sync_chunk_daily/all_sync_log_mdv_vn2.csv", DataFrame)

Row,ic,day,offset_1hz_s,offset_total_s,corr,vn_coverage,n_beams,nan_frac_vn2,status
,Int64,Date,Float64,Float64,Float64,Float64,Int64,Float64,String15
1,175,2024-04-29,-1.02,-0.62,0.4472,0.9922,537,0.0,ok
2,176,2024-04-29,-1.02,-0.62,0.5398,0.9935,537,0.0,ok
3,177,2024-04-29,-1.02,-0.67,0.7065,0.9934,518,0.0,ok
4,178,2024-04-29,-1.02,-0.57,0.4981,0.9928,537,0.0,ok
5,179,2024-04-29,-1.02,-0.62,0.8406,0.9915,537,0.0,ok
6,180,2024-04-29,-1.02,-0.67,0.4624,0.993,537,0.0,ok
7,181,2024-04-29,-1.02,-0.57,0.5598,0.9931,537,0.0,ok
8,182,2024-04-29,-1.02,-0.57,0.6904,0.9932,537,0.0,ok
9,183,2024-04-29,0.0,-0.5,0.5576,0.9919,518,0.0,ok


## Sign convention of `offset_total_s` (s)

```pseudocode
vn2_aligned[t] = vn2[t − offset_total_s] ≈ mdv[t]
```
VN timestamps are ~+0.85 s BEHIND lidar timestamps. VN data from EARLIER times need to be shifted up to align with the lidar.

The sign is defined by shift_signal_linear (line 469):

VN2 is shifted with final_offset to match mdv, so after shifting: vn2_shifted[t] = vn2[t − offset_total_s] ≈ mdv[t].

The lag equation is:
```
Vn2[t-δ] ≈ mdv[t]
δ = offset_total_s
```

Typical values are `offset_total_s` ≈ −0.85 s, so:
`Vn2[t+0.85s]≈mdv[t]`
meaning the VN clock lags the lidar clock by ~0.85 s — VN timestamps are about 0.85 s behind

In [ ]:
# plot an overview of sync performance

sync_ok = synclog[:, :status] .== "ok"
synclog[:, :ic]
#offset_total_s	corr	vn_coverage
fltm(x; missingval=-9999.0) = ( isapprox(x, missingval) ? NaN : x )

clf()
subplot(2,1,1)
plot(dtime_st[ic],  -0.04 .+zeros(size(ic)),   marker=".", markersize=2, linestyle="none", label="chunks")
plot(dtime_st[icvn], 1.04 .+zeros(size(icvn)), marker=".", markersize=2, linestyle="none", label="chunks w/ VN")

plot(dtime_st[synclog[:, :ic]],  sync_ok, marker=".", markersize=2, linestyle="none", label="sync ok")

plot(dtime_st[synclog[:, :ic]], -fltm.(synclog[:, :offset_total_s]), marker=".", markersize=2, linestyle="none", label="offset (s)")
plot(dtime_st[synclog[:, :ic]],  fltm.(synclog[:, :corr]), marker=".", markersize=1, linestyle="none", label="corr")
plot(Vn[:vndt][1:20*60:end],  1.08 .+ 0*Vn[:VelNED2][1:20*60:end], marker=".", markersize=1, linestyle="none", label="VelNED2")
ylim([-0.1, 1.1])
ylabel("offset (s) and corr")
legend(frameon=false)
ax = gca()
ax.xaxis.set_major_formatter(PythonPlot.matplotlib.dates.DateFormatter("%m.%d"))
PythonPlot.matplotlib.pyplot.setp(ax.get_xticklabels(), rotation=60) #, ha="right")
title("Sync between stare mean Doppler velocity and VN2 heave")
display("offset>0 <==> VN clock lags; synced by sampling VN earlier")

for fmt in ["png", "pdf", "eps", "svg"][1:1]
    savefig("sync_availability.$(fmt)") # overview of sync performance
end
gcf()


While technically improved, lag of less than 1 s explains why no offset refinement was needed to sync timeseries. The prior lag offset I provided before was close and worked. Now we have additional refinement, and confidence that that refinement was small.

In [55]:
Vn

Dict{Symbol, Vector} with 22 entries:
  :LinAcc1  => Float32[0.28455, -0.563547, 0.545289, -0.615101, 0.663306, -0.17…
  :Roll     => Float32[-0.684141, -0.711255, -0.714359, -0.761043, -0.767432, -…
  :Latitude => Float32[16.2633, 16.2633, 16.2633, 16.2633, 16.2633, 16.2633, 16…
  :vndt     => [DateTime("2024-04-29T05:25:42.128"), DateTime("2024-04-29T05:25…
  :time     => [DateTime("2023-03-29T13:17:44"), DateTime("2023-03-29T13:17:44"…
  :Yaw      => Float32[-162.59, -162.574, -162.565, -162.549, -162.541, -162.53…
  :VelNED1  => Float32[5.05222, 5.04132, 5.05353, 5.04639, 4.93381, 4.93558, 4.…
  :MagNED2  => Float32[0.09548, 0.100129, 0.09543, 0.095481, 0.093838, 0.09533,…
  :Altitude => Float32[-43.986, -43.986, -43.986, -43.986, -44.212, -44.212, -4…
  :Quat1    => Float32[0.005692, 0.005927, 0.005941, 0.006337, 0.006396, 0.0065…
  :VelNED2  => Float32[-0.280547, -0.279673, -0.27213, -0.271012, -0.272294, -0…
  :MagNED0  => Float32[0.234275, 0.237392, 0.23345, 0.236116, 0.233466,

In [ ]:
# fractions of good data
length(icvn) / length(ic) # 0.91 of chunks have VN data
sum(sync_ok) / length(sync_ok) # 0.47 of chunks with VN have good sync
# VN data missing or unsynced May 18-June 1.

0.47431412262935

In [ ]:
# load VN heave aligned to one chunk's lidar times from daily sync NC
vn2_aligned = load_vn2_aligned(ic, ist, ien, LidarDt["dtime"])